In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [ ]:
import os

for root, dirs, files in os.walk("../data"):
    print(root)
    for f in files:
        print("  ", f)

In [ ]:
calendar = pd.read_csv("../data/raw/calendar.csv")

sales = pd.read_csv(
    "../data/raw/sales_train_validation.csv"
)

prices = pd.read_csv(
    "../data/raw/sell_prices.csv"
)


In [ ]:
calendar.head()

In [ ]:
sales.head()

In [ ]:
prices.head()

In [ ]:
print(sales['item_id'].nunique())
print(sales['store_id'].nunique())
print(sales['state_id'].nunique())
print(sales['dept_id'].nunique())
print(sales['cat_id'].nunique())
sales[['item_id','dept_id','cat_id','store_id','state_id']].head()

In [ ]:
print("Calendar:", calendar.shape)
print("Sales:", sales.shape)
print("Prices:", prices.shape)


In [ ]:
sales.iloc[:, 6:].values.min()

In [ ]:
sales.iloc[:, 6:].values.max()

In [ ]:
(sales.iloc[:, 6:] == 0).sum().sum()

In [ ]:
product = sales.iloc[0]

print(product.head(15))

product_sales = product.iloc[6:]

print(product_sales.head())

import matplotlib.pyplot as plt

plt.figure(figsize=(15, 5))
plt.plot(product_sales.values)
plt.title(product['item_id'])
plt.show()

In [ ]:
# Working only on one store as the dataset is too big after melting the sales.csv
# Because if we melt the whole sales dataset then it will create 59 million rows which will be difficult fot the current system to handle

sales = sales[sales["store_id"] == 'CA_1']
print(sales.shape)
# Convert sales.csv from  Wide to Long
sales_long = pd.melt(
    sales,
    id_vars=[
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id"
    ],
    var_name="d",
    value_name="sales"
)

print(sales_long.shape)
sales_long.head()

In [ ]:
# Merging sales_long and Calendar to make a direct relationship between them

sales_long = sales_long.merge(
    calendar,
    on="d",
    how="left",
    validate="many_to_one"
)
print(sales_long.shape)
sales_long.head()

In [ ]:
sales_long.isnull().sum().sort_values(ascending=False)

In [ ]:
# Merging prices with sales_long now
sales_long = sales_long.merge(
    prices,
    on=[
        "store_id",
        "item_id",
        "wm_yr_wk"
    ],
    how="left",
    validate="many_to_one"
)
print(sales_long.shape)

In [ ]:
sales_long["sell_price"].isna().sum()

In [ ]:
sales_long.sample(10)

In [ ]:
sales_long = sales_long.sort_values(
    by=["item_id", "date"]
).reset_index(drop=True)

In [ ]:
sales_long.to_csv("../data/processed/model_data.csv", index=False)